## Error distributions

In this section I will look at the NRMSE error distributions produced by the four selected hierarchies, similar to how it was done with the so called base hierarchies. I will look at the measurements/plots which were highlights in the base hierarchies case.

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import numpy as np
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy, BinaryTemporalHierarchy, FullTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

In [ ]:
em = LoadedErrorMetrics('metrics/binary_hier_errors.csv')
inc = pd.read_csv('metrics/base_hier_inconsistency.csv', index_col=0)

sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()
bth = BinaryTemporalHierarchy()
fth = FullTemporalHierarchy()

error_metric = 'nrmse'
em = FilteredErrorMetrics(em, MI(error_metric='nrmse'))

diffs = MergedErrorMetrics([
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=fth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=bth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=sth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=mth)),
])

nrmse_em, nrmse_diffs = em, diffs

### Statistics

First I will look at the statistics of the reconciled forecast errors per hierarchy and granularity.

In [ ]:
nrmse_em.get_error_stats(MI(granularity=MONTHS, forecast_type=RECONCILED_FORECASTS))

Looking at monthly granularity error statistics, it is apparent how big the differences are in the means. `Full` hierarchy achieved the best value, being half as much as `Semantic`'s mean, and also being smaller than `Minimal`'s mean, which we know is not distorted by super extreme outliers. `Full` itself did not introduce any extreme outliers, as it is apparent by the maximum value. This produces an undistorted variance value, which is smaller than `Minimal`'s.

`Full` performs better than `Binary` at all percentiles, also achieving a global minimum with q75 and q90 percentiles. The median stayed the smallest for `Semantic` hierarchy, but `Full` is not lacking much, and is better than `Minimal`'s.

The worst performance out of these hierarchies is still `Minimal`, which also lost it's advantage of being the only hierarch not producing extreme outliers.

In [ ]:
nrmse_em.get_error_stats(MI(granularity=MONTHS, forecast_type=RECONCILED_FORECASTS),
                         df_filter=lambda df: df[df < np.quantile(df.values.flatten(), 0.9999)])

When sorting out the extreme outliers, we more reliably compare the means and variances. Still `Full` has the smallest mean monthly error, and also the smallest variance.

In [ ]:
nrmse_em.get_error_stats(MI(granularity=DAYS, forecast_type=RECONCILED_FORECASTS))

Looking at daily granularity error statistics, we can see that the two new hierarchies did introduce extreme outliers, so the means and variances are not comparable. Note that these outliers' magnitudes are several orders of magnitude smaller still.

From the medians, it's visible that the two new hierarchies did not worsen the performance much when looking at the medians. There is very little difference between the median of `Full` and `Binary`. Their q75 and q90 percentiles are smaller than either `Minimal`'s or `Semantic`'s, which suggests generally smaller big errors. Where these hierarchies are significantly lacking, are the smaller percentiles, where they achieve significantly bigger values, meaning bigger small errors.

In [ ]:
nrmse_em.get_error_stats(MI(granularity=DAYS, forecast_type=RECONCILED_FORECASTS),
                         df_filter=lambda df: df[df < np.quantile(df.values.flatten(), 0.9999)])

When sorting out the super extreme outliers, we can get some more meaningful insights from the means and variances.

After filtering we can see that a total of 6 values were sorted out. When looking at the means, `Full` hierarchy performs best. Though there is not a huge difference between the means, it still seems like having more aggregation levels decreases mean error, but increases median error at bottom level granularity. The variances are also the smallest with `Full` hierarchy, with `Binary` being a close second.

### QQ plots

Next we will look at QQ plots per hierarchy comparing the distributions of daily and monthly errors.

In [ ]:
for h in (mth, sth, bth, fth):
    plots.plot_error_qq(nrmse_em, MI(hierarchy=h, forecast_type=RECONCILED_FORECASTS, granularity=DAYS),
                                  MI(hierarchy=h, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS),
                                  outlier_threshold=0.99).show()

The overall shape of the plots for `Full` and `Binary` match the shape of `Semantic`'s, so nothing interesting can be derived.

### Histograms

Next I will look at histograms of the error distribution

In [ ]:
plots.plot_error_distribution(nrmse_em,
                              MetricsIndex.indeces_from_product(hierarchies=[mth, sth, bth, fth], forecast_types=[RECONCILED_FORECASTS], granularities=[MONTHS]) +
                              [MI(forecast_type=BASE_FORECASTS, hierarchy=mth, granularity=MONTHS)],
                              outlier_threshold=0.97)

When looking at the distributions at monthly granularity, and we compare `Full` with `Semantic`, we can see that the biggest difference is that `Full` has a much smaller tail, although a bit less minimal errors too. `Binary` behaves very similar compared to `Semantic`, but when compared to `Full`, it's visibly worse.

When comparing `Full` with `Minimal` (reconciled), it's advantage is very apparent, with small errors being more regular, and a much shorter tail. When comparing `Full` with the base forecast errors, we can see that by reconciliation the errors mostly became smaller, but the tail actually increased (even with `Full` hierarcy).

What we see here is consistent with what we saw about the percentiles of the monthly errors.

In [ ]:
plots.plot_error_distribution(nrmse_em,
                              MetricsIndex.indeces_from_product(hierarchies=[mth, sth, bth, fth], forecast_types=[RECONCILED_FORECASTS], granularities=[DAYS]) +
                              [MI(forecast_type=BASE_FORECASTS, hierarchy=mth, granularity=DAYS)],
                              outlier_threshold=0.97)

When looking at the distributions at daily granulariy, and we compare `Full` with `Minimal` (reconciled), we can see that `Minimal` has significantly more smaller errors, while it has a longer tail. The same is true for `Binary`. The same effects can be said when comparing to `Semantic`, just less extreme.

### Conclusions
The median values across the four hierarchies were relatively close at both granularities, so the differences between hierarchies are more visible in the means, variances, and upper percentiles. `Full` emerged as the strongest overall performer in this section, while `Minimal` remained the weakest.

**Full**
- Pros
    - When looking at monthly granularity reconciled forecast errors, this hierarchy had the lowest mean and variance (both raw and after filtering extreme outliers), and the lowest q75 and q90 percentiles, across all hierarchies.
    - When looking at daily granularity reconciled forecast errors (after filtering extreme outliers), this hierarchy had the lowest mean and variance, and the lowest q75 and q90 percentiles, across all hierarchies.
- Cons
    - When looking at monthly granularity reconciled forecast errors, this hierarchy had a worse median than `Semantic`, though still better than `Minimal`.
    - When looking at daily granularity reconciled forecast errors, this hierarchy had bigger q10 and q25 percentiles than all other hierarchies, meaning larger small errors.

**Likely trends** (to be investigated in subsequent sections)
- Improving with more levels:
    - Monthly mean and variance
    - Daily mean and variance
    - Monthly and daily q90
- Worsening with more levels:
    - Daily median
    - Monthly median

Highlights:
- Error statistics at both granularities